# ML-08 — Capstone Modeling Lane

This notebook clusters pages into descriptive, operationally useful groups. K-Means is unsupervised: it fits centroids and assigns observations to them; it does not predict a ground-truth label or estimate causal impact.

## 1. Data contract and method

The model uses K-Means because no validated archetype labels are available. The active-corpus contract is `impressions_90d >= 10` and `content_age_days >= 90`; `avg_position == 0` is removed because it represents missing rank data, not a rank of zero.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

possible_paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
    '/content/content_refresh_anonymized.csv',
]
data_path = next((path for path in possible_paths if os.path.exists(path)), None)
if data_path is None:
    raise FileNotFoundError('Could not locate content_refresh_anonymized.csv')

df_raw = pd.read_csv(data_path)
contract_mask = (df_raw['impressions_90d'] >= 10) & (df_raw['content_age_days'] >= 90)
df_clean = df_raw.loc[contract_mask & (df_raw['avg_position'] > 0)].copy()

print(f'Model corpus: {len(df_clean):,} pages across {df_clean["client_id"].nunique()} pseudonymized clients')
print(f'Excluded for no ranking data: {(contract_mask & (df_raw["avg_position"] == 0)).sum():,} pages')


## 2. Final feature selection

The leakage-safe candidate set contained 11 variables. The final model retains five dimensions that cover reach, ranking, click efficiency, freshness, and engagement without duplicating the primary clustering dimensions.

| Candidate | Final | Reason for decision |
| --- | --- | --- |
| `impressions_log` | Yes | Reach |
| `clicks_log` | No | Strongly overlaps with impressions (`r` is shown below) |
| `sessions_log` | No | Strongly overlaps with impressions (`r` is shown below) |
| `word_count_log` | No | Content attribute; substantial missingness and not a behavioral signal |
| `staleness_log` | Yes | Freshness |
| `avg_position` | Yes | Ranking |
| `ctr` | Yes | Click efficiency |
| `engagement_rate` | Yes | User behavior |
| `scroll_rate` | No | Additional engagement signal rather than a distinct objective dimension |
| `content_age_days` | No | Less direct than time since last update for this objective |
| `update_ratio` | No | Derived measure that overlaps with staleness |

In [ ]:
candidate_audit = pd.DataFrame({
    'impressions_log': np.log1p(df_clean['impressions_90d']),
    'clicks_log': np.log1p(df_clean['clicks_90d']),
    'sessions_log': np.log1p(df_clean['sessions_90d']),
    'word_count_log': np.log1p(df_clean['word_count'].fillna(0)),
    'staleness_log': np.log1p(df_clean['days_since_last_update']),
    'avg_position': df_clean['avg_position'],
    'ctr': df_clean['ctr'],
    'engagement_rate': df_clean['engagement_rate'],
    'scroll_rate': df_clean['scroll_rate'],
    'content_age_days': df_clean['content_age_days'],
    'update_ratio': np.clip(
        df_clean['days_since_last_update'] / (df_clean['content_age_days'] + 1), 0, 1
    ),
})

overlap_check = candidate_audit.corr().loc[
    ['impressions_log', 'staleness_log', 'engagement_rate'],
    ['clicks_log', 'sessions_log', 'update_ratio', 'scroll_rate'],
]
display(overlap_check.round(3))
display(pd.DataFrame({'missing_share': df_clean[['word_count', 'scroll_rate']].isna().mean()}).round(3))

model_features = [
    'impressions_log', 'avg_position', 'ctr', 'staleness_log', 'engagement_rate'
]
X = candidate_audit[model_features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## 3. Select K before naming clusters

K is compared from 2 through 8 using inertia, silhouette, Davies–Bouldin, cluster sizes, stability, and interpretability. The operational desire for a certain number of personas is not used as the selection rule.

In [ ]:
k_results = []
sample_idx = np.arange(0, len(X_scaled), 10)
for k in range(2, 9):
    candidate = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = candidate.fit_predict(X_scaled)
    k_results.append({
        'k': k,
        'inertia': candidate.inertia_,
        'silhouette': silhouette_score(X_scaled[sample_idx], labels[sample_idx]),
        'davies_bouldin': davies_bouldin_score(X_scaled, labels),
        'smallest_cluster': pd.Series(labels).value_counts().min(),
        'largest_cluster': pd.Series(labels).value_counts().max(),
    })

k_results = pd.DataFrame(k_results)
display(k_results.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(k_results['k'], k_results['inertia'], marker='o')
axes[0].set(title='Inertia (elbow)', xlabel='k')
axes[1].plot(k_results['k'], k_results['silhouette'], marker='o')
axes[1].set(title='Silhouette', xlabel='k')
axes[2].plot(k_results['k'], k_results['davies_bouldin'], marker='o')
axes[2].set(title='Davies–Bouldin (lower is better)', xlabel='k')
plt.tight_layout()
plt.show()


### K decision

For this extract, `K=5` is retained because the experiment table provides the strongest silhouette among the tested values while preserving compact, stable, non-empty clusters. Re-run this comparison if the data contract or observation window changes.

In [ ]:
final_k = 5
seeds = [0, 42, 100, 2026]
base_labels = None
stability_results = []
for seed in seeds:
    candidate = KMeans(n_clusters=final_k, random_state=seed, n_init=20)
    labels = candidate.fit_predict(X_scaled)
    if base_labels is None:
        base_labels = labels
        ari = np.nan
    else:
        ari = adjusted_rand_score(base_labels, labels)
    stability_results.append({
        'seed': seed,
        'silhouette': silhouette_score(X_scaled[sample_idx], labels[sample_idx]),
        'ARI_vs_seed_0': ari,
    })

stability_df = pd.DataFrame(stability_results)
display(stability_df.round(3))


## 4. Fit, hold out clients, and interpret clusters

The grouped check fits centroids on reference clients, assigns held-out-client pages to those centroids, and evaluates held-out cluster coherence. It is not classification accuracy. K-Means emits arbitrary numeric IDs; the names below are human interpretations assigned after inspecting the profiles.

In [ ]:
fold_results = []
gkf = GroupKFold(n_splits=5)
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, groups=df_clean['client_id']), start=1):
    fold_scaler = StandardScaler()
    X_train = fold_scaler.fit_transform(X.iloc[train_idx])
    X_test = fold_scaler.transform(X.iloc[test_idx])
    fold_model = KMeans(n_clusters=final_k, random_state=42, n_init=20).fit(X_train)
    test_labels = fold_model.predict(X_test)
    test_sample = np.arange(0, len(X_test), 10)
    fold_results.append({
        'fold': fold,
        'held_out_clients': df_clean.iloc[test_idx]['client_id'].nunique(),
        'held_out_pages': len(test_idx),
        'held_out_silhouette': silhouette_score(X_test[test_sample], test_labels[test_sample]),
    })

holdout_coherence = pd.DataFrame(fold_results)
display(holdout_coherence.round(3))

kmeans = KMeans(n_clusters=final_k, random_state=42, n_init=20)
df_clean['cluster'] = kmeans.fit_predict(X_scaled)
cluster_names = {
    0: 'Stale, High-Reach',
    1: 'Current, High-Reach',
    2: 'High-Engagement',
    3: 'Low-Visibility',
    4: 'High-CTR, Low-Reach',
}
df_clean['archetype'] = df_clean['cluster'].map(cluster_names)

cluster_profiles = df_clean.groupby('archetype').agg(
    pages=('content_id', 'size'),
    share_of_corpus=('content_id', lambda s: len(s) / len(df_clean) * 100),
    median_impressions=('impressions_90d', 'median'),
    median_position=('avg_position', 'median'),
    median_ctr=('ctr', 'median'),
    median_staleness=('days_since_last_update', 'median'),
    median_engagement=('engagement_rate', 'median'),
).round(2)
display(cluster_profiles.sort_values('pages', ascending=False))


## 5. Decision log

| Decision | Evidence | Result |
| --- | --- | --- |
| Use `log1p` for reach and staleness | Distribution audit shows a heavy right tail | Accepted |
| Standardize features | K-Means uses Euclidean distance | Accepted |
| Remove leakage/product outputs | Decision-time and privacy audit | Accepted |
| Use five final features | Redundancy, relevance, and missingness review | Accepted |
| Choose `K=5` | K comparison, cluster sizes, seed stability, and interpretation | Accepted for this extract |
| Group client evaluation | Prevents same-client observations in fit and holdout | Accepted |
| Name clusters | Profile inspection | Post-hoc human interpretation |

## Self-check

- [x] K was compared before the final number of archetypes was selected.
- [x] Stability and held-out client coherence are calculated from code.
- [x] Cluster names are explicitly distinguished from model output.
- [x] No client names, URLs, or raw queries are displayed.